# Dark Pattern Detector — Two-Stage DistilBERT Training

**Pipeline:** scraped text -> sentence segmentation -> Stage 1 (binary: dark pattern y/n) -> Stage 2 (category, only run on Stage-1 positives) -> JSON detections with confidence + offsets.

**Run order:** just run all cells top to bottom. Upload `dataset.csv` when prompted in the cell below.

**Runtime:** Runtime -> Change runtime type -> GPU (T4 is fine).


In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn spacy
!python -m spacy download en_core_web_sm -q


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME = "distilbert-base-uncased"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## 1. Load data
Upload `dataset.csv` (the cleaned/merged file with columns `page_id, text, label, Pattern Category`).

In [ ]:
from google.colab import files

if not os.path.exists("dataset.csv"):
    uploaded = files.upload()  # pick dataset.csv
    fname = list(uploaded.keys())[0]
    if fname != "dagent_dataset.csv":
        os.rename(fname, "dataset.csv")

df = pd.read_csv("dataset.csv")
df = df.dropna(subset=["text", "label", "Pattern Category"]).reset_index(drop=True)
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].reset_index(drop=True)

print(df.shape)
print(df["label"].value_counts())
print(df["Pattern Category"].value_counts())
df.head()


## 2. Grouped train/val/test split

Splitting by `page_id` (not by row) so sentences from the same page never appear in both train and test -- otherwise the model can "memorize" a page's style rather than generalizing, and validation accuracy will be inflated.

Split: 70% train / 15% val / 15% test, grouped by `page_id`.


In [ ]:
def group_split(df, group_col, test_size, seed):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx_a, idx_b = next(gss.split(df, groups=df[group_col]))
    return df.iloc[idx_a].reset_index(drop=True), df.iloc[idx_b].reset_index(drop=True)

train_df, temp_df = group_split(df, "page_id", test_size=0.30, seed=SEED)
val_df, test_df = group_split(temp_df, "page_id", test_size=0.50, seed=SEED)

print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)
print("Any page_id overlap train/val:", set(train_df.page_id) & set(val_df.page_id))
print("Any page_id overlap train/test:", set(train_df.page_id) & set(test_df.page_id))
print("Any page_id overlap val/test:", set(val_df.page_id) & set(test_df.page_id))


## 3. Stage 1 -- Binary classifier (dark pattern y/n)

Fine-tunes DistilBERT on `label` (0/1). This is the cheap first-pass filter in your cascade -- only text that this model flags as positive goes on to Stage 2 (category) and, in your deployed system, potentially to the LLM layer for low-confidence cases.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(pdf, label_col, label2id=None):
    d = pdf[["text", label_col]].rename(columns={label_col: "labels"}).copy()
    if label2id is not None:
        d["labels"] = d["labels"].map(label2id)
    return Dataset.from_pandas(d.reset_index(drop=True))

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

stage1_train = to_hf_dataset(train_df, "label").map(tokenize_fn, batched=True)
stage1_val   = to_hf_dataset(val_df, "label").map(tokenize_fn, batched=True)
stage1_test  = to_hf_dataset(test_df, "label").map(tokenize_fn, batched=True)

collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
import evaluate as hf_evaluate
acc_metric = hf_evaluate.load("accuracy")
f1_metric = hf_evaluate.load("f1")

def compute_metrics_binary(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

stage1_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label={0: "not_dark_pattern", 1: "dark_pattern"},
    label2id={"not_dark_pattern": 0, "dark_pattern": 1},
).to(device)

stage1_args = TrainingArguments(
    output_dir="./stage1_out",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=20,
    report_to="none",
    seed=SEED,
)

stage1_trainer = Trainer(
    model=stage1_model,
    args=stage1_args,
    train_dataset=stage1_train,
    eval_dataset=stage1_val,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics_binary,
)

stage1_trainer.train()


In [ ]:
# Held-out test set evaluation
preds = stage1_trainer.predict(stage1_test)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred, target_names=["not_dark_pattern", "dark_pattern"]))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


## 4. Stage 2 -- Category classifier

Trained **only on rows where `label == 1`**, predicting `Pattern Category` (Scarcity, Social Proof, Urgency, Misdirection, Obstruction, Sneaking, Forced Action).

**Caveat to remember:** Obstruction (27), Sneaking (12), and Forced Action (4) are extremely small classes. We use class weighting so the loss doesn't ignore them, but with only 4 examples of Forced Action, don't expect reliable recall on that class yet -- it needs more labeled data. This is a legitimate, explainable limitation, not a bug in the pipeline.


In [ ]:
cat_train_df = train_df[train_df.label == 1].reset_index(drop=True)
cat_val_df   = val_df[val_df.label == 1].reset_index(drop=True)
cat_test_df  = test_df[test_df.label == 1].reset_index(drop=True)

le = LabelEncoder()
le.fit(df.loc[df.label == 1, "Pattern Category"])
label2id = {c: i for i, c in enumerate(le.classes_)}
id2label = {i: c for c, i in label2id.items()}
print(label2id)

stage2_train = to_hf_dataset(cat_train_df, "Pattern Category", label2id).map(tokenize_fn, batched=True)
stage2_val   = to_hf_dataset(cat_val_df, "Pattern Category", label2id).map(tokenize_fn, batched=True)
stage2_test  = to_hf_dataset(cat_test_df, "Pattern Category", label2id).map(tokenize_fn, batched=True)

# class weights (inverse frequency) to counter the severe imbalance
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(label2id)),
    y=cat_train_df["Pattern Category"].map(label2id).values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print(dict(zip(le.classes_, class_weights.tolist())))


In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics_multiclass(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

stage2_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label2id), id2label=id2label, label2id=label2id,
).to(device)

stage2_args = TrainingArguments(
    output_dir="./stage2_out",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=20,
    report_to="none",
    seed=SEED,
)

stage2_trainer = WeightedTrainer(
    model=stage2_model,
    args=stage2_args,
    train_dataset=stage2_train,
    eval_dataset=stage2_val,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics_multiclass,
)

stage2_trainer.train()


In [ ]:
preds2 = stage2_trainer.predict(stage2_test)
y_pred2 = np.argmax(preds2.predictions, axis=-1)
y_true2 = preds2.label_ids

print(classification_report(
    y_true2, y_pred2,
    labels=list(range(len(le.classes_))),
    target_names=le.classes_,
    zero_division=0,
))
print("Confusion matrix (rows=true, cols=pred):\n", le.classes_)
print(confusion_matrix(y_true2, y_pred2, labels=list(range(len(le.classes_)))))


## 5. Save both models
Saves locally in Colab, then zips + downloads. Copy these into your inference service later.

In [ ]:
stage1_trainer.save_model("./dark_pattern_stage1_binary")
tokenizer.save_pretrained("./dark_pattern_stage1_binary")

stage2_trainer.save_model("./dark_pattern_stage2_category")
tokenizer.save_pretrained("./dark_pattern_stage2_category")

with open("./dark_pattern_stage2_category/label_map.json", "w") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, indent=2)

!zip -qr dark_pattern_models.zip dark_pattern_stage1_binary dark_pattern_stage2_category
from google.colab import files as colab_files
colab_files.download("dark_pattern_models.zip")


## 6. Inference pipeline

Matches the JSON contract we designed for the extension:

```json
{
  "source_text": "...",
  "detections": [
    {"sentence": "...", "category": "Scarcity", "confidence": 0.94, "start_offset": 0, "end_offset": 22}
  ]
}
```

Steps: sentence-segment the scraped block (spaCy, handles "Dr.", "$19.99", "e.g." correctly -- naive `.split('.')` does not) -> Stage 1 binary on each sentence -> for positives, Stage 2 category -> assemble detections list with true character offsets into the original block.


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

BINARY_THRESHOLD = 0.5   # tune on your val set; raise to reduce false positives
CATEGORY_THRESHOLD = 0.0 # report top category regardless; raise to require confident category too

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

@torch.no_grad()
def classify_binary(sentences):
    enc = tokenizer(sentences, truncation=True, max_length=128, padding=True, return_tensors="pt").to(device)
    logits = stage1_model(**enc).logits.cpu().numpy()
    probs = np.array([softmax(row) for row in logits])
    return probs  # shape (n, 2), col 1 = dark pattern prob

@torch.no_grad()
def classify_category(sentences):
    enc = tokenizer(sentences, truncation=True, max_length=128, padding=True, return_tensors="pt").to(device)
    logits = stage2_model(**enc).logits.cpu().numpy()
    probs = np.array([softmax(row) for row in logits])
    return probs  # shape (n, num_categories)

def detect_dark_patterns(source_text: str):
    doc = nlp(source_text)
    sents = [s for s in doc.sents if s.text.strip()]
    if not sents:
        return {"source_text": source_text, "detections": []}

    sent_texts = [s.text.strip() for s in sents]
    bin_probs = classify_binary(sent_texts)

    flagged_idx = [i for i, p in enumerate(bin_probs) if p[1] >= BINARY_THRESHOLD]
    detections = []
    if flagged_idx:
        flagged_texts = [sent_texts[i] for i in flagged_idx]
        cat_probs = classify_category(flagged_texts)
        for local_i, global_i in enumerate(flagged_idx):
            span = sents[global_i]
            cat_id = int(np.argmax(cat_probs[local_i]))
            cat_conf = float(cat_probs[local_i][cat_id])
            if cat_conf >= CATEGORY_THRESHOLD:
                detections.append({
                    "sentence": sent_texts[global_i],
                    "category": id2label[cat_id],
                    "confidence": round(cat_conf, 4),
                    "binary_confidence": round(float(bin_probs[global_i][1]), 4),
                    "start_offset": span.start_char,
                    "end_offset": span.end_char,
                })

    return {"source_text": source_text, "detections": detections}


In [ ]:
# quick smoke test -- replace with real scraped text once the extension is wired up
sample = (
    "Premium Wireless Headphones. "
    "Only 2 left in stock! "
    "14 people are viewing this item right now. "
    "Free shipping on orders over $50. "
    "Sale ends in 00:04:59. "
    "By continuing you agree to receive marketing emails, which you can unsubscribe from at any time."
)

result = detect_dark_patterns(sample)
print(json.dumps(result, indent=2))


## Next steps

- **Threshold tuning**: use the val set predictions (not test) to pick `BINARY_THRESHOLD` -- favor higher precision if false positives will annoy users, higher recall if you'd rather over-flag.
- **Deployment (Approach B)**: wrap `detect_dark_patterns` in a FastAPI endpoint, containerize, deploy behind your extension's backend; only call the external LLM for sentences with low `category` confidence (tie-breaker layer) or to generate the human-readable explanation.
- **Approach C (on-device)**: export both models to ONNX (`optimum` library's `ORTModelForSequenceClassification`) and run via `transformers.js` / `onnxruntime-web` in-browser -- much more realistic than shipping a Docker container to end users.
- **Tail-class data**: Obstruction/Sneaking/Forced Action need more labeled examples before Stage 2 will be trustworthy on them in production -- worth scraping/labeling more or looking for supplementary public datasets to merge in.
